# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()

title = metadata.get('name')
description = metadata.get('description')
print(f"{title}: {description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

For all entities (record sets, fields, columns, etc.), we reference them by their `@id`.

Let's list all record sets in the dataset:

In [ ]:
# List the available record sets by @id and name
record_sets = dataset.metadata.record_sets

record_set_ids = []
for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']} | Name: {rs.get('name', '<No name>')}")
    record_set_ids.append(rs['@id'])
    # List fields in this record set
    fields = rs.get('field', [])
    if fields:
        print("  Fields:")
        for fld in fields:
            print(f"    Field @id: {fld['@id']} | Name: {fld.get('name', '<No name>')} | DataType: {fld.get('dataType', '<No dataType>')}")
    else:
        print("  No fields listed in this record set.")

# Show one record from each record set
for rs_id in record_set_ids:
    print(f"\nSample record from RecordSet {rs_id}:")
    records = dataset.records(record_set=rs_id)
    try:
        print(next(records))
    except StopIteration:
        print("No records available for this RecordSet.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Use the record set and field `@id`s from the overview above. For demonstration, we'll extract all available record sets.

In [ ]:
# Extract data from each record set
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"RecordSet {record_set_id}: Loaded {len(df)} records.")
    print(f"Columns (@id): {df.columns.tolist()}")
    if len(df) > 0:
        display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.
We will demonstrate these steps using one record set and numeric field referenced by their `@id`.

Let's select a record set containing the GAD-7, PHQ-9, or PSQ scores (mental health assessment scores).
First, we identify the relevant field `@id`s. We'll filter and process the field representing the GAD-7 score as an example.

In [ ]:
# Choose first record set for demonstration purposes
main_rs_id = record_set_ids[0] if record_set_ids else None
main_df = dataframes.get(main_rs_id, pd.DataFrame())

# Find a numeric field, e.g., GAD-7 score
gad7_field_id = None
group_field_id = None

if main_rs_id is not None:
    fields_info = [fld for fld in next(rs for rs in record_sets if rs['@id']==main_rs_id).get('field', [])]
    for fld in fields_info:
        dtype = fld.get('dataType', '')
        name = fld.get('name', '').lower()
        if 'gad' in name and (dtype.lower() == 'integer' or dtype.lower() == 'float'):
            gad7_field_id = fld['@id']
        if 'gender' in name:
            group_field_id = fld['@id']

    # If GAD-7 score not found, fallback to any numeric field
    if gad7_field_id is None:
        for fld in fields_info:
            dtype = fld.get('dataType', '')
            if dtype.lower() == 'integer' or dtype.lower() == 'float':
                gad7_field_id = fld['@id']
                break
    # If group field not found, fallback to any string categorical field
    if group_field_id is None:
        for fld in fields_info:
            dtype = fld.get('dataType', '')
            name = fld.get('name', '').lower()
            if dtype.lower() == 'text' or dtype.lower() == 'string':
                group_field_id = fld['@id']
                break

    # Perform numeric filtering and normalization
    if gad7_field_id and gad7_field_id in main_df.columns:
        numeric_field = gad7_field_id
        threshold = 10
        filtered_df = main_df[main_df[numeric_field].astype(float) > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field].astype(float) - filtered_df[numeric_field].astype(float).mean()) / filtered_df[numeric_field].astype(float).std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"].copy()])

        # Group by categorical field if present
        if group_field_id and group_field_id in main_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field].mean().reset_index()
            print(f"Grouped data by {group_field_id} (mean {numeric_field}):")
            display(grouped_df)
    else:
        print("No numeric (GAD-7) field found for EDA in this record set.")
else:
    print("No available record set for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll plot the distribution of the selected numeric field (e.g., GAD-7 score) and its relation to a categorical field (e.g., gender), using the field `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_rs_id and gad7_field_id and gad7_field_id in main_df.columns:
    # Histogram
    plt.figure(figsize=(8,4))
    sns.histplot(main_df[gad7_field_id].astype(float), bins=20, kde=True)
    plt.title(f"Distribution of {gad7_field_id}")
    plt.xlabel(gad7_field_id)
    plt.ylabel("Count")
    plt.show()

    # Boxplot by group field
    if group_field_id and group_field_id in main_df.columns:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=main_df[group_field_id], y=main_df[gad7_field_id].astype(float))
        plt.title(f"{gad7_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(gad7_field_id)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset contains survey records from Kilifi County, Kenya, with fields covering demographic attributes and mental health indicators (e.g., GAD-7 scores).
- Data loaded and referenced by Croissant `@id` for each entity, ensuring FAIR reproducibility.
- Numeric fields can be filtered, normalized, and grouped by categorical demographic attributes, facilitating basic exploratory analysis.
- Visualization reveals the distribution and potential group differences in mental health scores.

This notebook demonstrates the use of `mlcroissant` for FAIR-compliant data exploration.

**For deeper ML tasks and modeling, proceed with further domain-specific analysis using the extracted DataFrames and Croissant entity IDs.**